In [ ]:
import time

import anyio

start_time = time.monotonic()


def log(message: str) -> None:
    elapsed = time.monotonic() - start_time
    print(f"[{elapsed:6.2f}s] {message}")


async def make_item_anyio(item: str = "coffee", brew: float = 2.0) -> str:
    await anyio.sleep(brew)
    return item

In [ ]:
start_time = time.monotonic()

with anyio.move_on_after(1.5) as scope:
    await make_item_anyio("slow drip", brew=0)

log(f"cancel_called={scope.cancel_called}")

In [ ]:
async def warm_up_station(task_status=anyio.TASK_STATUS_IGNORED):
    log("station: warming up")
    await make_item_anyio("warmup", brew=0.5)
    task_status.started("station-1")
    log("station: now serving")
    await make_item_anyio("espresso", brew=0.5)
    log("station: espresso served")


async with anyio.create_task_group() as group:
    station = await group.start(warm_up_station)
    log(f"main: {station} is ready — placing orders")

In [ ]:
async def brew_with_cleanup():
    try:
        log("brewing slow drip")
        await make_item_anyio("slow drip", brew=5.0)
    finally:
        with anyio.CancelScope(shield=True):
            log("cleanup: rinsing (shielded from cancel)")
            await make_item_anyio("rinse", brew=0.5)
            log("cleanup: rinse done")


with anyio.move_on_after(1.0):
    await brew_with_cleanup()

log("main: cancelled at 1.0s — but the shielded rinse still finished")

In [ ]:
async def barista(receive):
    async for order in receive:
        log(f"barista: making {order}")
        await make_item_anyio(order, brew=0.5)
        log(f"barista: {order} ready")
    log("barista: stream closed — clocking out")


async def front_desk(send):
    async with send:
        for order in ("espresso", "latte", "mocha"):
            log(f"desk: ordering {order}")
            await send.send(order)


send, receive = anyio.create_memory_object_stream(max_buffer_size=1)
async with anyio.create_task_group() as group:
    group.start_soon(barista, receive)
    group.start_soon(front_desk, send)